## MIMII train → Stage1 encoder/quantizer codebook usage

This notebook passes the full **MIMII training dataset** (all machine types + machine ids) through a **Stage 1 pretrained VQ-VAE-2 model** and computes:

- codebook index frequencies for **coarse** and **fine** index maps
- empirical probabilities \(p(k)\)
- entropy \(H = -\sum_k p(k)\log_2 p(k)\) (bits)


In [1]:
from __future__ import annotations

import os
from pathlib import Path
import sys

# Resolve repo root (audDSR) so `src` imports work from this notebook
_here = Path.cwd().resolve()
REPO_ROOT = _here
for p in [_here, *_here.parents]:
    if (p / "src").is_dir():
        REPO_ROOT = p
        break
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import torch
from torch.utils.data import DataLoader

from src.data.dataset import DCASE2020Task2LogMelDataset
from src.models.vq_vae.autoencoders import VQ_VAE_2Layer
from src.utils.checkpoint_compat import migrate_vq_vae_state_dict


In [2]:
# ---- Paths / runtime ----
PROJECT_ROOT = Path.cwd().resolve().parent.parent  # notebooks/comm_stack -> project root
if (PROJECT_ROOT / "src").exists() and str(PROJECT_ROOT) not in os.sys.path:
    os.sys.path.insert(0, str(PROJECT_ROOT))

# Set this to your dataset root. If you already export DATA_PATH, this will use it.
DATA_PATH = Path(os.environ.get("DATA_PATH", "/mnt/ssd/LaCie/dcase2020_task2/dcase2020_task2_dev_dataset"))
CKPT_DIR = PROJECT_ROOT / "checkpoints"

# Point this at your pretrained stage1 checkpoint.
# Default matches the path used in notebooks/stage1_reconstruction_analysis.ipynb.
STAGE1_CKPT = (
    CKPT_DIR
    / "stage1"
    / "ToyCar+ToyConveyor+fan+pump+slider+valve"
    / "checkpoints_trainandtest_emb128_64_hid256_128_fine1024_coarse512_iter20000_bs256_stage1_ToyCar+ToyConveyor+fan+pump+slider+valve_best.pt"
)
if not STAGE1_CKPT.exists():
    raise FileNotFoundError(
        f"Stage1 checkpoint not found at {STAGE1_CKPT}.\n"
        f"Set STAGE1_CKPT manually or ensure checkpoints are present under {CKPT_DIR / 'stage1'}."
    )

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BATCH_SIZE = 64
NUM_WORKERS = 0

print("PROJECT_ROOT:", PROJECT_ROOT)
print("DATA_PATH:", DATA_PATH)
print("STAGE1_CKPT:", STAGE1_CKPT)
print("DEVICE:", DEVICE)


PROJECT_ROOT: /home/lucash/Documents/NTUST/Research/papers/semantic-communication-networks/audDSR
DATA_PATH: /mnt/ssd/LaCie/dcase2020_task2/dcase2020_task2_dev_dataset
STAGE1_CKPT: /home/lucash/Documents/NTUST/Research/papers/semantic-communication-networks/audDSR/checkpoints/stage1/ToyCar+ToyConveyor+fan+pump+slider+valve/checkpoints_trainandtest_emb128_64_hid256_128_fine1024_coarse512_iter20000_bs256_stage1_ToyCar+ToyConveyor+fan+pump+slider+valve_best.pt
DEVICE: cuda


In [4]:
# ---- Build MIMII training dataset (all machine types + ids) ----
# We detect machine types by scanning DATA_PATH/*/train
machine_types = sorted(
    [p.name for p in DATA_PATH.iterdir() if p.is_dir() and (p / "train").exists()]
)
print("Detected machine types:", machine_types)

# For the question: use ONLY training split.
# (Dataset supports include_test=True, but we want training dataset only.)
train_ds = DCASE2020Task2LogMelDataset(
    root=str(DATA_PATH),
    machine_types=machine_types,
    include_test=False,
)

train_loader = DataLoader(
    train_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=(DEVICE.type == "cuda"),
)

len(train_ds), next(iter(train_loader))[0].shape

Detected machine types: ['ToyCar', 'ToyConveyor', 'fan', 'pump', 'slider', 'valve']
DCASE2020Task2LogMelDataset: ToyCar+ToyConveyor+fan+pump+slider+valve | 20119 spectrograms (T→320), shape (20119, 1, 128, 320) | IDs: ['ToyCar__id_01', 'ToyCar__id_02', 'ToyCar__id_03', 'ToyCar__id_04', 'ToyConveyor__id_01', 'ToyConveyor__id_02', 'ToyConveyor__id_03', 'fan__id_00', 'fan__id_02', 'fan__id_04', 'fan__id_06', 'pump__id_00', 'pump__id_02', 'pump__id_04', 'pump__id_06', 'slider__id_00', 'slider__id_02', 'slider__id_04', 'slider__id_06', 'valve__id_00', 'valve__id_02', 'valve__id_04', 'valve__id_06'] | 3.30 GB in RAM


(20119, torch.Size([64, 1, 128, 320]))

In [5]:
# ---- Load Stage1 pretrained VQ-VAE-2 ----
ckpt = torch.load(STAGE1_CKPT, map_location="cpu", weights_only=True)

vq_vae = VQ_VAE_2Layer(
    hidden_channels=(ckpt["hidden_channels_coarse"], ckpt["hidden_channels_fine"]),
    num_residual_layers=ckpt["num_residual_layers"],
    num_embeddings=(ckpt["num_embeddings_coarse"], ckpt["num_embeddings_fine"]),
    embedding_dim=(ckpt["embedding_dim_coarse"], ckpt["embedding_dim_fine"]),
    commitment_cost=0.25,
    decay=0.95,
)
state = dict(ckpt["model_state_dict"])
migrate_vq_vae_state_dict(state)
vq_vae.load_state_dict(state)
vq_vae = vq_vae.eval().to(DEVICE)

num_embeddings_coarse = int(ckpt["num_embeddings_coarse"])
num_embeddings_fine = int(ckpt["num_embeddings_fine"])
print("num_embeddings_coarse:", num_embeddings_coarse)
print("num_embeddings_fine:", num_embeddings_fine)


num_embeddings_coarse: 512
num_embeddings_fine: 1024


In [6]:
def entropy_bits_from_counts(counts: torch.Tensor) -> float:
    """counts: (K,) int64 on CPU."""
    total = counts.sum().item()
    if total == 0:
        return 0.0
    p = counts.double() / float(total)
    p = p[p > 0]
    return float(-(p * torch.log2(p)).sum().item())


def probs_from_counts(counts: torch.Tensor) -> torch.Tensor:
    total = counts.sum().item()
    if total == 0:
        return torch.zeros_like(counts, dtype=torch.float64)
    return counts.double() / float(total)


In [7]:
# ---- Run full dataset through encoder+quantizer (indices) and count usage ----
counts_coarse = torch.zeros(num_embeddings_coarse, dtype=torch.int64)
counts_fine = torch.zeros(num_embeddings_fine, dtype=torch.int64)

total_positions_coarse = 0
total_positions_fine = 0

with torch.inference_mode():
    for step, batch in enumerate(train_loader):
        x, _, _ = batch  # x: (B, 1, n_mels, T)
        x = x.to(DEVICE, non_blocking=True)

        idx_coarse, idx_fine = vq_vae.encode_to_indices(x)
        # idx_*: (B, H, W)

        flat_c = idx_coarse.reshape(-1).to("cpu", non_blocking=True)
        flat_f = idx_fine.reshape(-1).to("cpu", non_blocking=True)

        counts_coarse += torch.bincount(flat_c, minlength=num_embeddings_coarse)
        counts_fine += torch.bincount(flat_f, minlength=num_embeddings_fine)

        total_positions_coarse += flat_c.numel()
        total_positions_fine += flat_f.numel()

        if (step + 1) % 50 == 0:
            print(
                f"[{step+1:>5d}/{len(train_loader)}] "
                f"positions coarse={total_positions_coarse:,} fine={total_positions_fine:,}"
            )

print("Done.")
print("Total positions coarse:", total_positions_coarse)
print("Total positions fine:", total_positions_fine)
print("Nonzero coarse indices:", int((counts_coarse > 0).sum().item()))
print("Nonzero fine indices:", int((counts_fine > 0).sum().item()))


[   50/315] positions coarse=2,048,000 fine=8,192,000
[  100/315] positions coarse=4,096,000 fine=16,384,000
[  150/315] positions coarse=6,144,000 fine=24,576,000
[  200/315] positions coarse=8,192,000 fine=32,768,000
[  250/315] positions coarse=10,240,000 fine=40,960,000
[  300/315] positions coarse=12,288,000 fine=49,152,000
Done.
Total positions coarse: 12876160
Total positions fine: 51504640
Nonzero coarse indices: 512
Nonzero fine indices: 1024


In [8]:
# ---- Empirical probabilities and entropy ----
probs_coarse = probs_from_counts(counts_coarse)
probs_fine = probs_from_counts(counts_fine)

H_coarse = entropy_bits_from_counts(counts_coarse)
H_fine = entropy_bits_from_counts(counts_fine)

print(f"Entropy (coarse) [bits]: {H_coarse:.6f}")
print(f"Entropy (fine)   [bits]: {H_fine:.6f}")

# Optional sanity checks
print("Sum p(coarse):", float(probs_coarse.sum().item()))
print("Sum p(fine):  ", float(probs_fine.sum().item()))


Entropy (coarse) [bits]: 8.620424
Entropy (fine)   [bits]: 9.461723
Sum p(coarse): 1.0
Sum p(fine):   1.0


In [9]:
# ---- Show top utilized indices ----

def topk_table(counts: torch.Tensor, k: int = 20):
    k = min(k, counts.numel())
    vals, idx = torch.topk(counts, k)
    probs = probs_from_counts(counts)
    rows = []
    for i in range(k):
        rows.append((int(idx[i].item()), int(vals[i].item()), float(probs[idx[i]].item())))
    return rows

print("Top coarse indices (idx, count, p):")
for r in topk_table(counts_coarse, k=20):
    print(r)

print("\nTop fine indices (idx, count, p):")
for r in topk_table(counts_fine, k=20):
    print(r)


Top coarse indices (idx, count, p):
(0, 418966, 0.03253811695412297)
(419, 69811, 0.005421725110591978)
(459, 68843, 0.005346547417863711)
(317, 66233, 0.00514384723395795)
(378, 64656, 0.0050213728316516725)
(198, 62981, 0.004891287464585715)
(256, 62658, 0.004866202346041056)
(490, 61879, 0.004805702942492171)
(503, 61799, 0.00479948991003529)
(424, 61154, 0.004749397335851682)
(324, 60443, 0.0046941790098911475)
(294, 60306, 0.004683539191808738)
(495, 60284, 0.004681830607883096)
(299, 59178, 0.0045959354341667084)
(427, 59084, 0.0045886351210298725)
(501, 58275, 0.004525805830309658)
(81, 56838, 0.0044142042348029225)
(156, 56803, 0.004411486033103037)
(388, 56504, 0.004388264824295442)
(491, 56233, 0.004367218176847756)

Top fine indices (idx, count, p):
(0, 1495591, 0.029037985703812316)
(1022, 163216, 0.003168957204632437)
(993, 159627, 0.0030992741624832247)
(1023, 157389, 0.0030558217667379095)
(1015, 156692, 0.0030422890054177644)
(1012, 155640, 0.0030218636612157662)
(1017,

In [ ]:
# ---- Entropy per machine_type ----
# Note: `machine_id` returned by the dataset is composite when multiple machine types are loaded:
#   "{machine_type}__{id_XX}". We'll parse machine_type from that.

from collections import defaultdict

counts_by_mt_coarse: dict[str, torch.Tensor] = {
    mt: torch.zeros(num_embeddings_coarse, dtype=torch.int64) for mt in machine_types
}
counts_by_mt_fine: dict[str, torch.Tensor] = {
    mt: torch.zeros(num_embeddings_fine, dtype=torch.int64) for mt in machine_types
}

total_pos_by_mt_coarse = defaultdict(int)
total_pos_by_mt_fine = defaultdict(int)

with torch.inference_mode():
    for step, batch in enumerate(train_loader):
        x, _, mids = batch  # mids: list[str]
        x = x.to(DEVICE, non_blocking=True)

        idx_coarse, idx_fine = vq_vae.encode_to_indices(x)  # (B, H, W)

        # Parse machine_type keys for this batch.
        mt_keys = [m.split("__", 1)[0] for m in mids]

        # Accumulate counts grouped by machine_type (one bincount per group).
        for mt in set(mt_keys):
            mask = torch.tensor([k == mt for k in mt_keys], dtype=torch.bool, device=idx_coarse.device)

            flat_c = idx_coarse[mask].reshape(-1).to("cpu", non_blocking=True)
            flat_f = idx_fine[mask].reshape(-1).to("cpu", non_blocking=True)

            counts_by_mt_coarse[mt] += torch.bincount(flat_c, minlength=num_embeddings_coarse)
            counts_by_mt_fine[mt] += torch.bincount(flat_f, minlength=num_embeddings_fine)

            total_pos_by_mt_coarse[mt] += flat_c.numel()
            total_pos_by_mt_fine[mt] += flat_f.numel()

        if (step + 1) % 50 == 0:
            print(f"[{step+1:>5d}/{len(train_loader)}] aggregated per-machine_type")

print("Done per-machine_type counting.")

In [ ]:
# ---- Summarize entropies per machine_type ----
rows = []
for mt in machine_types:
    c_c = counts_by_mt_coarse[mt]
    c_f = counts_by_mt_fine[mt]
    rows.append(
        {
            "machine_type": mt,
            "H_coarse_bits": entropy_bits_from_counts(c_c),
            "H_fine_bits": entropy_bits_from_counts(c_f),
            "nonzero_coarse": int((c_c > 0).sum().item()),
            "nonzero_fine": int((c_f > 0).sum().item()),
            "positions_coarse": int(total_pos_by_mt_coarse[mt]),
            "positions_fine": int(total_pos_by_mt_fine[mt]),
        }
    )

# Pretty display (pandas if available, otherwise plain print)
try:
    import pandas as pd

    df = pd.DataFrame(rows).sort_values("machine_type")
    display(df)
except Exception:
    for r in rows:
        print(r)